In [1]:
# =============================================
# Cardiovascular Risk Prediction - Data Loading
# =============================================

# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, BaggingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import logging

# Suppress warnings for clean output
warnings.filterwarnings("ignore")
logging.getLogger("pyarrow").setLevel(logging.ERROR)
os.environ["PYTHONWARNINGS"] = "ignore"

# ----------------------------
# Load dataset
# ----------------------------
file_path = "gemini_cardio.csv"  # Path to your CSV file
data = pd.read_csv(file_path)

# Replace spaces in column names with underscores for easier access
data.columns = [col.replace(" ", "_") for col in data.columns]

# ----------------------------
# Display dataset info
# ----------------------------
print("First 5 rows of the dataset:")
print(data.head())

print("\nDataset information:")
print(data.info())


<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject


First 5 rows of the dataset:
   age  sex  chest_pain_type  resting_bp_s  cholesterol  fasting_blood_sugar  \
0   54    1                3           132          221                    0   
1   48    1                4           128          195                    0   
2   62    0                2           140          240                    1   
3   41    1                1           118          188                    0   
4   59    1                4           152          215                    0   

   resting_ecg  max_heart_rate  exercise_angina  oldpeak  ST_slope  target  
0            1             142                0      0.8         2       1  
1            0             155                1      1.2         2       0  
2            2             128                0      0.5         1       1  
3            0             168                0      0.0         1       0  
4            1             115                1      2.1         3       1  

Dataset information:
<class

In [2]:
# =============================================
# Feature Identification and Scaling
# =============================================

# Define categorical and continuous features
categorical_features = ['sex', 'chest_pain_type', 'fasting_blood_sugar',
                        'resting_ecg', 'exercise_angina', 'ST_slope', 'target']

continuous_features = ['age', 'resting_bp_s', 'cholesterol', 'max_heart_rate', 'oldpeak']

# Display feature types
print("Categorical features:", categorical_features)
print("Continuous features:", continuous_features)

# ----------------------------
# Feature Scaling
# ----------------------------
from sklearn.preprocessing import StandardScaler

# Columns to be scaled
numerical_cols = ['age', 'resting_bp_s', 'cholesterol', 'max_heart_rate', 'oldpeak']

# Initialize StandardScaler
scaler = StandardScaler()

# Fit and transform only numerical columns
data[numerical_cols] = scaler.fit_transform(data[numerical_cols])

print("\nScaled dataset preview:")
print(data.head())

# ----------------------------
# Train-Test Split
# ----------------------------
from sklearn.model_selection import train_test_split

# Split data: 80% training, 20% testing
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Display dataset sizes
print(f"\nTotal data points: {len(data)}")
print(f"Training data points: {len(train_data)}")
print(f"Testing data points: {len(test_data)}")


Categorical features: ['sex', 'chest_pain_type', 'fasting_blood_sugar', 'resting_ecg', 'exercise_angina', 'ST_slope', 'target']
Continuous features: ['age', 'resting_bp_s', 'cholesterol', 'max_heart_rate', 'oldpeak']

Scaled dataset preview:
        age  sex  chest_pain_type  resting_bp_s  cholesterol  \
0  0.247250    1                3     -0.055421     0.198205   
1 -0.373461    1                4     -0.322508    -0.159541   
2  1.074865    0                2      0.478755     0.459634   
3 -1.097625    1                1     -0.990228    -0.255857   
4  0.764510    1                4      1.280018     0.115648   

   fasting_blood_sugar  resting_ecg  max_heart_rate  exercise_angina  \
0                    0            1       -0.161410                0   
1                    0            0        0.351003                1   
2                    1            2       -0.713239                0   
3                    0            0        0.863415                0   
4            

In [3]:
# =============================================
# AdaBoost Model Training and Evaluation
# =============================================

from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
from sklearn.model_selection import KFold, GridSearchCV, learning_curve
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# Feature and target separation
# ----------------------------
X_train = train_data.drop(columns=['target'])
y_train = train_data['target']
X_test = test_data.drop(columns=['target'])
y_test = test_data['target']

# ----------------------------
# Initial AdaBoost model
# ----------------------------
adaboost_model = AdaBoostClassifier(n_estimators=50, random_state=42)
adaboost_model.fit(X_train, y_train)

# Predictions on test set
y_pred = adaboost_model.predict(X_test)

# Model accuracy
accuracy_adaboost = accuracy_score(y_test, y_pred)
print(f"Initial AdaBoost Model Accuracy: {accuracy_adaboost:.4f}")

# ----------------------------
# K-Fold Cross-Validation
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
adaboost_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    adaboost_model.fit(X_fold_train, y_fold_train)
    y_fold_pred = adaboost_model.predict(X_fold_val)
    adaboost_scores.append(accuracy_score(y_fold_val, y_fold_pred))

print(f"K-Fold Mean Accuracy (Initial Model): {np.mean(adaboost_scores):.4f}")

# ----------------------------
# Hyperparameter Tuning
# ----------------------------
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 1]
    # 'algorithm' parametresi buradan kaldırıldı
}

grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

# Best hyperparameters and score

print(f"Best Hyperparameters: {grid_search.best_params_}")

print(f"Best Cross-Validated Accuracy: {grid_search.best_score_:.4f}")

# ----------------------------
# Evaluate best model on test set
# ----------------------------
best_adaboost_model = grid_search.best_estimator_
y_pred = best_adaboost_model.predict(X_test)

accuracy_best = accuracy_score(y_test, y_pred)
print(f"Test Accuracy of Best AdaBoost Model: {accuracy_best:.4f}")

# ----------------------------
# K-Fold Cross-Validation for Best Model
# ----------------------------
adaboost_scores = []
for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    best_adaboost_model.fit(X_fold_train, y_fold_train)
    y_fold_pred = best_adaboost_model.predict(X_fold_val)
    adaboost_scores.append(accuracy_score(y_fold_val, y_fold_pred))

print(f"K-Fold Mean Accuracy (Best Model): {np.mean(adaboost_scores):.4f}")

# ----------------------------
# Model Performance Metrics
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))



plt.tight_layout()
plt.show()


Initial AdaBoost Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial Model): 0.9688
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best Hyperparameters: {'learning_rate': 0.1, 'n_estimators': 100}
Best Cross-Validated Accuracy: 0.9750
Test Accuracy of Best AdaBoost Model: 1.0000
K-Fold Mean Accuracy (Best Model): 0.9563

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [4]:
# =============================================
# Gradient Boosting Machine (GBM) Model Training and Evaluation
# =============================================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# Initial GBM model
# ----------------------------
gbm_model = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, random_state=42)
gbm_model.fit(X_train, y_train)

# Predictions on test set
y_pred_gbm = gbm_model.predict(X_test)

# Model accuracy
accuracy_gbm = accuracy_score(y_test, y_pred_gbm)
print(f"Initial GBM Model Accuracy: {accuracy_gbm:.4f}")

# ----------------------------
# K-Fold Cross-Validation for GBM
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
gbm_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    gbm_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_gbm = gbm_model.predict(X_fold_val)
    gbm_scores.append(accuracy_score(y_fold_val, y_fold_pred_gbm))

print(f"K-Fold Mean Accuracy (Initial GBM): {np.mean(gbm_scores):.4f}")

# ----------------------------
# Hyperparameter Tuning with GridSearchCV
# ----------------------------
gbm_param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'subsample': [0.8, 1.0]
}

gbm_kf = KFold(n_splits=5, shuffle=True, random_state=42)

gbm_grid_search = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid=gbm_param_grid,
    scoring='accuracy',
    cv=gbm_kf,
    n_jobs=-1
)

gbm_grid_search.fit(X_train, y_train)

# Best hyperparameters and cross-validated accuracy
print(f"Best Hyperparameters: {gbm_grid_search.best_params_}")
print(f"Best Cross-Validated Accuracy: {gbm_grid_search.best_score_:.4f}")

# ----------------------------
# Evaluate best GBM model on test set
# ----------------------------
best_gbm_model = gbm_grid_search.best_estimator_
y_pred_best_gbm = best_gbm_model.predict(X_test)
accuracy_best_gbm = accuracy_score(y_test, y_pred_best_gbm)
print(f"Test Accuracy of Best GBM Model: {accuracy_best_gbm:.4f}")

# K-Fold Cross-Validation for Best GBM Model
kfold_scores = cross_val_score(best_gbm_model, X_train, y_train, cv=gbm_kf, scoring='accuracy', n_jobs=-1)
print("K-Fold Cross-Validation Accuracy Scores:", kfold_scores)
print(f"Mean K-Fold Accuracy: {kfold_scores.mean():.4f}")

# ----------------------------
# Model Performance Metrics
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_gbm))

plt.tight_layout()
plt.show()


Initial GBM Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial GBM): 0.9437
Best Hyperparameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best Cross-Validated Accuracy: 0.9688
Test Accuracy of Best GBM Model: 1.0000
K-Fold Cross-Validation Accuracy Scores: [0.96875 0.96875 0.9375  1.      0.96875]
Mean K-Fold Accuracy: 0.9688

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [5]:
# =============================================
# XGBoost Model Training and Evaluation
# =============================================

from xgboost import XGBClassifier
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score, learning_curve
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# Initial XGBoost model
# ----------------------------
xgb_model = XGBClassifier(n_estimators=50, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Predictions on test set
y_pred_xgb = xgb_model.predict(X_test)

# Model accuracy
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f"Initial XGBoost Model Accuracy: {accuracy_xgb:.4f}")

# ----------------------------
# K-Fold Cross-Validation for XGBoost
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
xgb_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    xgb_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_xgb = xgb_model.predict(X_fold_val)
    xgb_scores.append(accuracy_score(y_fold_val, y_fold_pred_xgb))

print(f"K-Fold Mean Accuracy (Initial XGBoost): {np.mean(xgb_scores):.4f}")

# ----------------------------
# Hyperparameter Tuning with GridSearchCV
# ----------------------------
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')

param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train)

# Best hyperparameters and accuracy
print(f"Best Hyperparameters: {grid_search.best_params_}")
print(f"Best Cross-Validated Accuracy: {grid_search.best_score_:.4f}")

# ----------------------------
# Evaluate best XGBoost model on test set
# ----------------------------
best_xgb_model = grid_search.best_estimator_
y_pred_best_xgb = best_xgb_model.predict(X_test)
accuracy_best_xgb = accuracy_score(y_test, y_pred_best_xgb)
print(f"Test Accuracy of Best XGBoost Model: {accuracy_best_xgb:.4f}")

# K-Fold Cross-Validation for Best XGBoost Model
kfold_scores_xgb = cross_val_score(best_xgb_model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print("K-Fold Cross-Validation Accuracy Scores:", kfold_scores_xgb)
print(f"Mean K-Fold Accuracy: {kfold_scores_xgb.mean():.4f}")

# ----------------------------
# Model Performance Metrics
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_xgb))


plt.tight_layout()
plt.show()


Initial XGBoost Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial XGBoost): 0.9563
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best Cross-Validated Accuracy: 0.9750
Test Accuracy of Best XGBoost Model: 1.0000
K-Fold Cross-Validation Accuracy Scores: [0.96875 1.      0.96875 1.      0.9375 ]
Mean K-Fold Accuracy: 0.9750

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [6]:
# =============================================
# LightGBM Model Training and Evaluation
# =============================================

from lightgbm import LGBMClassifier
from sklearn.model_selection import KFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
import warnings
import os
import logging

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*pyarrow.*")
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger("pyarrow").setLevel(logging.ERROR)

# ----------------------------
# Initial LightGBM model
# ----------------------------
lgbm_model = LGBMClassifier(n_estimators=50, learning_rate=0.1, random_state=42, verbose=-1)
lgbm_model.fit(X_train, y_train)

# Predictions on test set
y_pred_lgbm = lgbm_model.predict(X_test)

# Model accuracy
accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
print(f"Initial LightGBM Model Accuracy: {accuracy_lgbm:.4f}")

# ----------------------------
# K-Fold Cross-Validation
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
lgbm_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    lgbm_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_lgbm = lgbm_model.predict(X_fold_val)
    lgbm_scores.append(accuracy_score(y_fold_val, y_fold_pred_lgbm))

print(f"K-Fold Mean Accuracy (Initial LightGBM): {np.mean(lgbm_scores):.4f}")

# ----------------------------
# Hyperparameter Tuning with GridSearchCV
# ----------------------------
param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 1.0]
}

lgbm_grid_search = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
lgbm_grid_search.fit(X_train, y_train)

print(f"Best Hyperparameters: {lgbm_grid_search.best_params_}")
print(f"Best Cross-Validated Accuracy: {lgbm_grid_search.best_score_:.4f}")

# ----------------------------
# Train Best LightGBM Model
# ----------------------------
best_lgbm_model = LGBMClassifier(**lgbm_grid_search.best_params_, random_state=42)
best_lgbm_model.fit(X_train, y_train)

y_pred_best_lgbm = best_lgbm_model.predict(X_test)
accuracy_best_lgbm = accuracy_score(y_test, y_pred_best_lgbm)
print(f"Test Accuracy of Best LightGBM Model: {accuracy_best_lgbm:.4f}")

# K-Fold Cross-Validation for Best Model
best_lgbm_scores = []
for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    best_lgbm_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_best_lgbm = best_lgbm_model.predict(X_fold_val)
    best_lgbm_scores.append(accuracy_score(y_fold_val, y_fold_pred_best_lgbm))

print(f"K-Fold Mean Accuracy (Best LightGBM): {np.mean(best_lgbm_scores):.4f}")

# ----------------------------
# Model Performance Metrics
# ----------------------------
from sklearn.metrics import classification_report

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_best_lgbm))


plt.tight_layout()
plt.show()


Initial LightGBM Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial LightGBM): 0.9688
Fitting 5 folds for each of 54 candidates, totalling 270 fits
Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50, 'subsample': 0.8}
Best Cross-Validated Accuracy: 0.9750
Test Accuracy of Best LightGBM Model: 1.0000
K-Fold Mean Accuracy (Best LightGBM): 0.9688

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [7]:
# =============================================
# Stochastic Gradient Boosting (SGB) Model
# =============================================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import KFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# Initial SGB model
# ----------------------------
sgb_model = GradientBoostingClassifier(
    n_estimators=50,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)
sgb_model.fit(X_train, y_train)

# Predictions on test set
y_pred_sgb = sgb_model.predict(X_test)

# Model accuracy
accuracy_sgb = accuracy_score(y_test, y_pred_sgb)
print(f"Stochastic Gradient Boosting Model Accuracy: {accuracy_sgb:.4f}")

# ----------------------------
# K-Fold Cross-Validation
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
sgb_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    # Train a temporary SGB model for each fold
    sgb_temp = GradientBoostingClassifier(n_estimators=50, learning_rate=0.1, subsample=0.8, random_state=42)
    sgb_temp.fit(X_fold_train, y_fold_train)
    
    y_fold_pred_sgb = sgb_temp.predict(X_fold_val)
    sgb_scores.append(accuracy_score(y_fold_val, y_fold_pred_sgb))

kfold_mean_sgb = np.mean(sgb_scores)
print(f"K-Fold Mean Accuracy (Initial SGB): {kfold_mean_sgb:.4f}")

# ----------------------------
# Hyperparameter Tuning
# ----------------------------
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150],
    'max_depth': [3, 5, 7],
    'subsample': [0.6, 0.8, 1.0]
}

sgb_model = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(
    sgb_model,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)

best_params = grid_search.best_params_
print(f"Best Hyperparameters: {best_params}")

# ----------------------------
# Train Best SGB Model
# ----------------------------
best_sgb_model = GradientBoostingClassifier(**best_params, random_state=42)
best_sgb_model.fit(X_train, y_train)

y_pred_best_sgb = best_sgb_model.predict(X_test)
best_accuracy_sgb = accuracy_score(y_test, y_pred_best_sgb)
print(f"Test Accuracy of Best SGB Model: {best_accuracy_sgb:.4f}")

# K-Fold Cross-Validation for optimized model
optimized_sgb_scores = []
for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    sgb_temp = GradientBoostingClassifier(**best_params, random_state=42)
    sgb_temp.fit(X_fold_train, y_fold_train)
    
    y_fold_pred_sgb = sgb_temp.predict(X_fold_val)
    optimized_sgb_scores.append(accuracy_score(y_fold_val, y_fold_pred_sgb))

kfold_mean_best_sgb = np.mean(optimized_sgb_scores)
print(f"K-Fold Mean Accuracy (Best SGB): {kfold_mean_best_sgb:.4f}")

# ----------------------------
# Classification Report, ROC and Learning Curve
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_sgb))


plt.tight_layout()
plt.show()


Stochastic Gradient Boosting Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial SGB): 0.9500
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters: {'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 100, 'subsample': 0.6}
Test Accuracy of Best SGB Model: 1.0000
K-Fold Mean Accuracy (Best SGB): 0.9750

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [8]:
# =============================================
# Random Forest (RF) Model
# =============================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np
import warnings
import os
import logging

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger("pyarrow").setLevel(logging.ERROR)

# ----------------------------
# Initial RF Model
# ----------------------------
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Predictions on test set
y_pred_rf = rf_model.predict(X_test)

# Model accuracy
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Model Accuracy: {accuracy_rf:.4f}")

# ----------------------------
# K-Fold Cross-Validation
# ----------------------------
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rf_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    rf_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_rf = rf_model.predict(X_fold_val)
    rf_scores.append(accuracy_score(y_fold_val, y_fold_pred_rf))

print(f"K-Fold Mean Accuracy (Initial RF): {np.mean(rf_scores):.4f}")

# ----------------------------
# Hyperparameter Tuning
# ----------------------------
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search_rf = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=2
)
grid_search_rf.fit(X_train, y_train)

best_params = grid_search_rf.best_params_
print(f"Best Hyperparameters: {best_params}")

# ----------------------------
# Train Best RF Model
# ----------------------------
best_rf_model = grid_search_rf.best_estimator_
best_rf_model.fit(X_train, y_train)

y_pred_best_rf = best_rf_model.predict(X_test)
accuracy_best_rf = accuracy_score(y_test, y_pred_best_rf)
print(f"Test Accuracy of Best RF Model: {accuracy_best_rf:.4f}")

# K-Fold CV for optimized RF
best_rf_scores = []
for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    best_rf_model.fit(X_fold_train, y_fold_train)
    y_fold_pred_best_rf = best_rf_model.predict(X_fold_val)
    best_rf_scores.append(accuracy_score(y_fold_val, y_fold_pred_best_rf))

print(f"K-Fold Mean Accuracy (Best RF): {np.mean(best_rf_scores):.4f}")

# ----------------------------
# Classification Report, ROC, Learning Curve
# ----------------------------
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))


plt.tight_layout()
plt.show()


Random Forest Model Accuracy: 1.0000
K-Fold Mean Accuracy (Initial RF): 0.9688
Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best Hyperparameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Test Accuracy of Best RF Model: 1.0000
K-Fold Mean Accuracy (Best RF): 0.9688

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



<Figure size 640x480 with 0 Axes>

In [9]:
# =================================================================
# Cardiovascular Risk Prediction - Random Forest & AdaBoost
# =================================================================

import warnings
import os
import logging

# Suppress binary incompatibility and future warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger("pyarrow").setLevel(logging.ERROR)

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import KFold, GridSearchCV, learning_curve
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------
# Model Configuration
# ----------------------------
# Define base estimator and AdaBoost container
rf_base = RandomForestClassifier(n_estimators=50, random_state=42)
rf_boost = AdaBoostClassifier(estimator=rf_base, n_estimators=50, random_state=42)

# ----------------------------
# Hyperparameter Tuning
# ----------------------------
# Searching for the optimal configuration before final evaluation
param_grid = {
    'estimator__n_estimators': [50, 100, 150],
    'estimator__max_depth': [3, 5, 10],
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.1, 1]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=rf_boost, 
    param_grid=param_grid, 
    cv=kf, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

# Execute the grid search on training data
grid_search.fit(X_train, y_train)

# Extracting the best estimator
best_model = grid_search.best_estimator_
best_params = grid_search.best_params_
best_score = grid_search.best_score_

# ----------------------------
# Model Evaluation
# ----------------------------
print("\n" + "="*40)
print("MODEL OPTIMIZATION RESULTS")
print("="*40)
print(f"Best Hyperparameters: {best_params}")
print(f"Best K-Fold Accuracy (Train Set): {best_score:.4f}")

# Generate predictions on the hold-out test set
y_pred_best = best_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred_best)

print(f"\nFinal Test Set Accuracy: {test_accuracy:.4f}")
print("\nClassification Report (Optimized Model):\n")
print(classification_report(y_test, y_pred_best))

# ----------------------------
# Stability Verification (K-Fold)
# ----------------------------
# Final cross-validation to ensure model stability
best_rf_boost_scores = []

for train_index, val_index in kf.split(X_train):
    X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
    y_fold_train, y_fold_val = y_train.iloc[train_index], y_train.iloc[val_index]
    
    best_model.fit(X_fold_train, y_fold_train)
    y_fold_pred = best_model.predict(X_fold_val)
    best_rf_boost_scores.append(accuracy_score(y_fold_val, y_fold_pred))

print(f"Mean K-Fold Accuracy (Stability Check): {np.mean(best_rf_boost_scores):.4f}")

# ----------------------------
# Visualization
# ----------------------------
plt.tight_layout()
plt.show()


Fitting 5 folds for each of 81 candidates, totalling 405 fits

MODEL OPTIMIZATION RESULTS
Best Hyperparameters: {'estimator__max_depth': 3, 'estimator__n_estimators': 50, 'learning_rate': 0.01, 'n_estimators': 50}
Best K-Fold Accuracy (Train Set): 0.9750

Final Test Set Accuracy: 1.0000

Classification Report (Optimized Model):

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Mean K-Fold Accuracy (Stability Check): 0.9750


<Figure size 640x480 with 0 Axes>

In [10]:
# -----------------------------
# META-LEARNING MODELS
# -----------------------------
# -----------------------------
## Super Learners (Stacking)
# -----------------------------

In [11]:
# ----------------------------
# Import Required Libraries
# ----------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import (AdaBoostClassifier, GradientBoostingClassifier, 
                              RandomForestClassifier, HistGradientBoostingClassifier)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from mlxtend.classifier import StackingCVClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ----------------------------
# Load Dataset
# ----------------------------
file_path = "gemini_cardio.csv"
df = pd.read_csv(file_path)

# Replace spaces in column names with underscores
df.columns = [col.replace(" ", "_") for col in df.columns]
print(df.head())

# Features (X) and target variable (y)
X = df.drop(columns=["target"])
y = df["target"]

# ----------------------------
# Define Base Learners
# ----------------------------
ada = AdaBoostClassifier(random_state=42)
gbm = GradientBoostingClassifier(random_state=42)
xgb = XGBClassifier(eval_metric="logloss", random_state=42)
lgb = LGBMClassifier(random_state=42, verbose=-1)
rf = RandomForestClassifier(random_state=42)
s_gbm = GradientBoostingClassifier(subsample=0.8, random_state=42)  # Stochastic Gradient Boosting
ada_rf = AdaBoostClassifier(estimator=RandomForestClassifier(random_state=42), 
                            random_state=42)

# ----------------------------
# Define Meta Learner
# ----------------------------
meta_learner = HistGradientBoostingClassifier(random_state=42)

# ----------------------------
# Hyperparameter Grids
# ----------------------------
param_grids = {
    'ada': {
        'n_estimators': [50, 100, 150],
        'learning_rate': [0.01, 0.1, 1]
        # 'algorithm' parametresi kaldırıldı (v1.6+ için)
    },
    'gbm': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 4, 5],
        'subsample': [0.8, 1.0]
    },
    'xgb': {
        'n_estimators': [50, 100, 150],
        'learning_rate': [0.01, 0.1, 0.2],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    },
    'lgb': {
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        'subsample': [0.8, 1.0]
    },
    's_gbm': {
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'n_estimators': [50, 100, 150],
        'max_depth': [3, 5, 7],
        'subsample': [0.6, 0.8, 1.0]
    },
    'rf': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'ada_rf': {
        'estimator__n_estimators': [50, 100, 150],
        'estimator__max_depth': [3, 5, 10],
        'n_estimators': [50, 100, 150],
        'learning_rate': [0.01, 0.1, 1]
    }
}

# ----------------------------
# Run GridSearchCV for Each Base Learner
# ----------------------------
grid_search_results = {}
for model_name, model in zip(
    ['ada', 'gbm', 'xgb', 'lgb', 'rf', 's_gbm', 'ada_rf'],
    [ada, gbm, xgb, lgb, rf, s_gbm, ada_rf]
):
    grid_search = GridSearchCV(
        estimator=model, 
        param_grid=param_grids[model_name], 
        cv=5, 
        n_jobs=-1, 
        verbose=0
    )
    grid_search.fit(X, y)
    grid_search_results[model_name] = grid_search.best_estimator_
    print(f"Best parameters for {model_name}: {grid_search.best_params_}")

# ----------------------------
# Build Stacking Model with Optimized Base Learners
# ----------------------------
stacking_clf = StackingCVClassifier(
    classifiers=[
        grid_search_results['ada'], grid_search_results['gbm'], grid_search_results['xgb'], 
        grid_search_results['lgb'], grid_search_results['rf'], grid_search_results['s_gbm'], 
        grid_search_results['ada_rf']
    ], 
    meta_classifier=meta_learner, 
    use_probas=True, 
    cv=5,  # 5-fold cross-validation
    random_state=42
)

# ----------------------------
# Train and Evaluate Stacking Model
# ----------------------------
stacking_clf.fit(X, y)
stacking_accuracy = stacking_clf.score(X, y)
print(f"Accuracy of optimized stacking model: {stacking_accuracy:.4f}")


   age  sex  chest_pain_type  resting_bp_s  cholesterol  fasting_blood_sugar  \
0   54    1                3           132          221                    0   
1   48    1                4           128          195                    0   
2   62    0                2           140          240                    1   
3   41    1                1           118          188                    0   
4   59    1                4           152          215                    0   

   resting_ecg  max_heart_rate  exercise_angina  oldpeak  ST_slope  target  
0            1             142                0      0.8         2       1  
1            0             155                1      1.2         2       0  
2            2             128                0      0.5         1       1  
3            0             168                0      0.0         1       0  
4            1             115                1      2.1         3       1  
Best parameters for ada: {'learning_rate': 0.1, 'n_estima

In [12]:
# =====================================================================
# Classification Report for Stacking Ensemble
# =====================================================================

from sklearn.metrics import classification_report, accuracy_score

# -----------------------------
# Predictions on Test Set
# -----------------------------
y_pred_stacking = stacking_clf.predict(X_test)

# -----------------------------
# Classification Report
# -----------------------------
# Print detailed classification metrics (precision, recall, f1-score)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stacking))

# -----------------------------
# Optional: Accuracy Score
# -----------------------------
accuracy = accuracy_score(y_test, y_pred_stacking)
print(f"Overall Accuracy: {accuracy:.4f}")



Classification Report:

              precision    recall  f1-score   support

           0       0.60      1.00      0.75        24
           1       0.00      0.00      0.00        16

    accuracy                           0.60        40
   macro avg       0.30      0.50      0.38        40
weighted avg       0.36      0.60      0.45        40

Overall Accuracy: 0.6000


In [13]:
# =====================================================================
# Define Optimized Base Learners
# =====================================================================

from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# -----------------------------
# Base Learners with Optimized Hyperparameters
# -----------------------------

# AdaBoost classifier (Düzeltildi)
ada = AdaBoostClassifier(
    learning_rate=1,
    n_estimators=50,
    random_state=42
)


# Gradient Boosting Machine (GBM)
gbm = GradientBoostingClassifier(
    learning_rate=0.2,
    max_depth=5,
    n_estimators=200,
    subsample=0.8,
    random_state=42
)

# XGBoost classifier
xgb = XGBClassifier(
    learning_rate=0.2,
    max_depth=5,
    n_estimators=100,
    subsample=1.0,
    eval_metric="logloss",
    random_state=42
)

# LightGBM classifier
lgb = LGBMClassifier(
    learning_rate=0.1,
    max_depth=5,
    n_estimators=200,
    subsample=0.8,
    random_state=42,
    verbose=-1
)

# Random Forest classifier
rf = RandomForestClassifier(
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    n_estimators=300,
    random_state=42
)

# Stochastic Gradient Boosting
s_gbm = GradientBoostingClassifier(
    learning_rate=0.1,
    max_depth=5,
    n_estimators=100,
    subsample=0.6,
    random_state=42
)

# AdaBoost with Random Forest as base estimator (Zaten doğru görünüyor ama kontrol edin)
ada_rf = AdaBoostClassifier(
    estimator=RandomForestClassifier(
        max_depth=10,
        n_estimators=100,
        random_state=42
    ),
    learning_rate=1,
    n_estimators=50,
    random_state=42
)


In [14]:
# =====================================================================
# Stacking Ensemble with Logistic Regression Meta Learner
# =====================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from mlxtend.classifier import StackingCVClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

# -----------------------------
# Prepare Features and Target
# -----------------------------
X = df.drop(columns=["target"])  # Independent variables
y = df["target"]                 # Dependent variable

# -----------------------------
# Define Meta Learner (Logistic Regression with L2 regularization)
# -----------------------------
meta_learner = make_pipeline(
    StandardScaler(),  # Standardize features
    LogisticRegression(
        penalty="l2", 
        C=1.0, 
        solver="lbfgs", 
        max_iter=500
    )
)

# -----------------------------
# Define Stacking Classifier
# -----------------------------
stacking_lr = StackingCVClassifier(
    classifiers=[ada, gbm, xgb, lgb, rf, s_gbm, ada_rf],  # Base models
    meta_classifier=meta_learner,                        # Meta learner
    use_probas=True,                                     # Use probabilistic predictions
    random_state=42
)

# -----------------------------
# 5-Fold Cross-Validation Accuracy
# -----------------------------
cv_scores = cross_val_score(stacking_lr, X, y, cv=5, scoring="accuracy")
print(f"Stacking Model 5-Fold Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# -----------------------------
# Fit Model on Full Dataset
# -----------------------------
stacking_lr.fit(X, y)
stacking_accuracy = stacking_lr.score(X, y)
print(f"Stacking Model Accuracy on Full Data: {stacking_accuracy:.4f}")

# -----------------------------
# Train-Test Split Evaluation
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
stacking_lr.fit(X_train, y_train)

# Predictions on test set
y_pred_stacking = stacking_lr.predict(X_test)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stacking))


Stacking Model 5-Fold Accuracy: 0.9750 ± 0.0274
Stacking Model Accuracy on Full Data: 1.0000

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [15]:
# =====================================================================
# Stacking Ensemble with XGBoost Meta Learner
# =====================================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier, HistGradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from mlxtend.classifier import StackingCVClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import warnings
import os
import logging

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
os.environ['PYTHONWARNINGS'] = 'ignore'
logging.getLogger("pyarrow").setLevel(logging.ERROR)

# -----------------------------
# Prepare Features and Target
# -----------------------------
X = df.drop(columns=["target"])  # Independent variables
y = df["target"]                 # Dependent variable

# -----------------------------
# Define Meta Learner (XGBoost)
# -----------------------------
meta_learner = XGBClassifier(
    n_estimators=100,
    eval_metric="logloss",
    random_state=42
)

# -----------------------------
# Define Stacking Classifier
# -----------------------------
stacking_xgb = StackingCVClassifier(
    classifiers=[ada, gbm, xgb, lgb, rf, s_gbm, ada_rf],  # Base models
    meta_classifier=meta_learner,                        # XGBoost as meta learner
    use_probas=True,                                     # Use probabilistic predictions
    cv=5,                                                # 5-Fold Cross Validation
    random_state=42
)

# -----------------------------
# 5-Fold Cross-Validation Accuracy
# -----------------------------
cv_scores = cross_val_score(stacking_xgb, X, y, cv=5, scoring="accuracy")
print(f"Stacking Model 5-Fold Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# -----------------------------
# Fit Model on Full Dataset
# -----------------------------
stacking_xgb.fit(X, y)
stacking_accuracy = stacking_xgb.score(X, y)
print(f"Stacking Model Accuracy on Full Data: {stacking_accuracy:.4f}")

# -----------------------------
# Train-Test Split Evaluation
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
stacking_xgb.fit(X_train, y_train)

# Predictions on test set
y_pred_stacking = stacking_xgb.predict(X_test)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_stacking))


Stacking Model 5-Fold Accuracy: 0.9800 ± 0.0187
Stacking Model Accuracy on Full Data: 1.0000

Classification Report:

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        16

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40



In [16]:
# -----------------------------
# META-LEARNING MODELS
# -----------------------------
# -----------------------------
## Comprehensive Optimization Blending 
# -----------------------------

In [17]:
# ----------------------------
# Import Required Libraries
# ----------------------------
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# ----------------------------
# Load and Preprocess Dataset
# ----------------------------
dataset_path = "gemini_cardio.csv"
df = pd.read_csv(dataset_path)

# Standardize column names by replacing spaces with underscores
df.columns = [col.replace(" ", "_") for col in df.columns]
print(df.head())

# Separate features and target
X = df.drop(columns=["target"]).values  # Convert features to NumPy array
y = df["target"].values

# ----------------------------
# Train-Test Split
# ----------------------------
# 80% training, 20% testing with stratification to preserve class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ----------------------------
# Define Base Learners
# ----------------------------
ada = AdaBoostClassifier(n_estimators=100, random_state=42)
gbm = GradientBoostingClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(n_estimators=100, eval_metric="logloss", random_state=42)
lgb = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
s_gbm = GradientBoostingClassifier(n_estimators=100, subsample=0.8, random_state=42)  # Stochastic Gradient Boosting
# 'base_estimator' yerine 'estimator' yazıyoruz
ada_rf = AdaBoostClassifier(
    estimator=RandomForestClassifier(n_estimators=50, random_state=42),
    n_estimators=100,
    random_state=42
)

# List of base models for blending
base_models = [
    ("ada", ada),
    ("gbm", gbm),
    ("xgb", xgb),
    ("lgb", lgb),
    ("rf", rf),
    ("s_gbm", s_gbm),
    ("ada_rf", ada_rf)
]

# ----------------------------
# Generate Base Model Predictions (5-Fold Blending)
# ----------------------------
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_predictions_proba = np.zeros((X_train.shape[0], len(base_models)))
test_predictions_proba = np.zeros((kf.get_n_splits(), X_test.shape[0], len(base_models)))

for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train[train_index], X_train[val_index]
    y_tr, y_val = y_train[train_index], y_train[val_index]
    
    for j, (name, model) in enumerate(base_models):
        # Train base model on the fold
        model.fit(X_tr, y_tr)
        
        # Predict probabilities on validation fold
        base_predictions_proba[val_index, j] = model.predict_proba(X_val)[:, 1]
        
        # Predict probabilities on test set for current fold
        test_predictions_proba[i, :, j] = model.predict_proba(X_test)[:, 1]

# Average test set predictions across folds
test_predictions_proba = test_predictions_proba.mean(axis=0)

# ----------------------------
# Train Meta-Model
# ----------------------------
meta_model = LogisticRegression()
meta_model.fit(base_predictions_proba, y_train)  # Fit on entire training set
meta_predictions_proba = meta_model.predict_proba(test_predictions_proba)[:, 1]

# ----------------------------
# Evaluate Model Performance
# ----------------------------
auc_score = roc_auc_score(y_test, meta_predictions_proba)
print(f"Blending Model AUC Score: {auc_score:.4f}")


   age  sex  chest_pain_type  resting_bp_s  cholesterol  fasting_blood_sugar  \
0   54    1                3           132          221                    0   
1   48    1                4           128          195                    0   
2   62    0                2           140          240                    1   
3   41    1                1           118          188                    0   
4   59    1                4           152          215                    0   

   resting_ecg  max_heart_rate  exercise_angina  oldpeak  ST_slope  target  
0            1             142                0      0.8         2       1  
1            0             155                1      1.2         2       0  
2            2             128                0      0.5         1       1  
3            0             168                0      0.0         1       0  
4            1             115                1      2.1         3       1  
Blending Model AUC Score: 1.0000


In [18]:
# ----------------------------
# Import Required Libraries
# ----------------------------
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, train_test_split, GridSearchCV
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings("ignore")  # Suppress warnings

# ----------------------------
# Load and Preprocess Dataset
# ----------------------------
dataset_path = "gemini_cardio.csv"
df = pd.read_csv(dataset_path)

# Standardize column names by replacing spaces with underscores
df.columns = [col.replace(" ", "_") for col in df.columns]
print(df.head())

# Separate features and target
X = df.drop(columns=["target"]).values
y = df["target"].values

# ----------------------------
# Train-Test Split
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ----------------------------
# Define Base Learners
# ----------------------------
base_models = [
    ("ada", AdaBoostClassifier(random_state=42)),
    ("gbm", GradientBoostingClassifier(random_state=42)),
    ("xgb", XGBClassifier(eval_metric="logloss", random_state=42)),
    ("lgb", LGBMClassifier(random_state=42, verbose=-1)),
    ("rf", RandomForestClassifier(random_state=42)),
    ("s_gbm", GradientBoostingClassifier(subsample=0.8, random_state=42)),
    ("ada_rf", AdaBoostClassifier(estimator=RandomForestClassifier(random_state=42), random_state=42))
]

# ----------------------------
# Define Hyperparameter Grids
# ----------------------------
param_grids = {
    'ada': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1.0]},
    'gbm': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5]},
    'xgb': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5]},
    'lgb': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5]},
    'rf': {'n_estimators': [50, 100, 200], 'max_depth': [3, 5, 10]},
    's_gbm': {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 0.5]},
    'ada_rf': {'n_estimators': [50, 100, 200], 'estimator__n_estimators': [50, 100]}
}

# ----------------------------
# 5-Fold Cross-Validation for Blending
# ----------------------------
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_predictions_proba = np.zeros((X_train.shape[0], len(base_models)))
test_predictions_proba = np.zeros((kf.get_n_splits(), X_test.shape[0], len(base_models)))

for i, (train_index, val_index) in enumerate(kf.split(X_train, y_train)):
    X_tr, X_val = X_train[train_index], X_train[val_index]
    y_tr, y_val = y_train[train_index], y_train[val_index]
    
    for j, (name, model) in enumerate(base_models):
        print(f"Tuning {name} model...")
        grid_search = GridSearchCV(model, param_grids[name], cv=3, scoring="roc_auc", n_jobs=-1)
        grid_search.fit(X_tr, y_tr)

        best_model = grid_search.best_estimator_
        print(f"Best Parameters for {name}: {grid_search.best_params_}")

        # Store validation probabilities
        base_predictions_proba[val_index, j] = best_model.predict_proba(X_val)[:, 1]
        # Store test probabilities for this fold
        test_predictions_proba[i, :, j] = best_model.predict_proba(X_test)[:, 1]

# Average test set predictions across folds
test_predictions_proba = test_predictions_proba.mean(axis=0)

# ----------------------------
# Train Meta-Model
# ----------------------------
meta_model = LogisticRegression()
meta_model.fit(base_predictions_proba, y_train)
meta_predictions_proba = meta_model.predict_proba(test_predictions_proba)[:, 1]

# ----------------------------
# Evaluate Model Performance
# ----------------------------
auc_score = roc_auc_score(y_test, meta_predictions_proba)
print(f"\nBlending Model AUC Score: {auc_score:.4f}")


   age  sex  chest_pain_type  resting_bp_s  cholesterol  fasting_blood_sugar  \
0   54    1                3           132          221                    0   
1   48    1                4           128          195                    0   
2   62    0                2           140          240                    1   
3   41    1                1           118          188                    0   
4   59    1                4           152          215                    0   

   resting_ecg  max_heart_rate  exercise_angina  oldpeak  ST_slope  target  
0            1             142                0      0.8         2       1  
1            0             155                1      1.2         2       0  
2            2             128                0      0.5         1       1  
3            0             168                0      0.0         1       0  
4            1             115                1      2.1         3       1  
Tuning ada model...
Best Parameters for ada: {'learning_r

In [22]:
# ============================================================
# Performance Evaluation of Blending Meta-Model
# ============================================================

# ----------------------------
# Import Required Libraries
# ----------------------------
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Generate predictions
# We use the meta_model to predict the final labels based on base model outputs
y_pred_labels = meta_model.predict(test_predictions_proba)

# ----------------------------
# Compute Evaluation Metrics
# ----------------------------
auc_score = roc_auc_score(y_test, meta_predictions_proba)
accuracy = accuracy_score(y_test, meta_model.predict(test_predictions_proba))
precision = precision_score(y_test, meta_model.predict(test_predictions_proba))
recall = recall_score(y_test, meta_model.predict(test_predictions_proba))
f1 = f1_score(y_test, meta_model.predict(test_predictions_proba))

# Print performance metrics
print(f"Blending Model AUC Score: {auc_score:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# ----------------------------
# 2) Classification Report
# ----------------------------
print("Classification Report:\n")
print(classification_report(y_test, y_pred_labels, digits=4))


Blending Model AUC Score: 1.0000
Accuracy: 0.9750
Precision: 0.9524
Recall: 1.0000
F1 Score: 0.9756
Classification Report:

              precision    recall  f1-score   support

           0     1.0000    0.9500    0.9744        20
           1     0.9524    1.0000    0.9756        20

    accuracy                         0.9750        40
   macro avg     0.9762    0.9750    0.9750        40
weighted avg     0.9762    0.9750    0.9750        40

[CV] END learning_rate=0.01, max_depth=3, n_estimators=50, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=150, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=150, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=5, n_estimators=50, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=5, n_estimators=50, subsample=1.0; total t